#モデル１

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.inspection import PartialDependenceDisplay
from sklearn.preprocessing import LabelEncoder
from statsmodels.stats.outliers_influence import variance_inflation_factor # Import VIF
from statsmodels.tools.tools import add_constant

# 1. データの読み込み
# ファイル名が異なる場合は適宜変更してください
df = pd.read_csv("new_combined_result_with_SVF_and_Bands.csv")

# 2. 前処理

# LandsatLSTの0を欠損値(NaN)に置換
if 'LandsatLST' in df.columns:
    df['LandsatLST'] = df['LandsatLST'].replace(0, np.nan)

# 日付の抽出とカテゴリID化
if '撮影日時' in df.columns:
    df['date_obj'] = pd.to_datetime(df['撮影日時']).dt.date
    le_date = LabelEncoder()
    df['date_id'] = le_date.fit_transform(df['date_obj'])

# sun_shade の数値化 (今回はモデルに使用しませんが、データセットには残しておきます)
if 'Shade' in df.columns:
    if df['Shade'].dtype == 'object':
        le_sun = LabelEncoder()
        df['Shade'] = le_sun.fit_transform(df['Shade'].astype(str))
    else:
        df['Shade'] = pd.to_numeric(df['Shade'], errors='coerce')

# Sentinel-2 バンドのスケール変換
s2_bands = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']
for col in s2_bands:
    if col in df.columns:
        if df[col].max() > 1:
            df[col] = df[col] / 10000.0

# 3. 特徴量エンジニアリング

# ★【変更点】South_NDVI_Shadow_Integrated は作成しない
# ★【変更点】sun_shade も説明変数に含めない

# --- 変数の設定 ---
base_cols = ['LandsatLST', 'NDVI', 'SVF'] # sun_shadeを削除
model_cols = base_cols + s2_bands

target_col = '平均温度__c'

# 欠損値を含む行を削除
available_cols = [c for c in model_cols + [target_col, 'date_id'] if c in df.columns]
df_clean = df.dropna(subset=available_cols)

print(f"分析対象データ数: {len(df_clean)}")

# --- 多重共線性(Multicollinearity)の確認 ---
print("\n--- 多重共線性の確認 ---")

# 相関行列の計算と可視化
plt.figure(figsize=(12, 10))
correlation_matrix = df_clean[model_cols].corr()
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap='coolwarm', square=True)
plt.tight_layout()
plt.savefig('model1_correlation_matrix.png')
plt.show()

# VIF (Variance Inflation Factor) の計算
X_vif = add_constant(df_clean[model_cols])
vif_data = pd.DataFrame()
vif_data["feature"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]

print("\nVariance Inflation Factor (VIF):")
print(vif_data)
print("-" * 30)


# 3. モデルの構築 (ランダムフォレスト)

X = df_clean[model_cols]
y = df_clean[target_col]
dates = df_clean['date_id']

# データを8:2に分割
X_train, X_test, y_train, y_test, dates_train, dates_test = train_test_split(
    X, y, dates, test_size=0.2, random_state=42
)

# 学習
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# 4. 検証と評価
y_pred = rf.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("-" * 30)
print(f"モデル評価結果 (Shade除外):")
print(f"RMSE: {rmse:.4f}")
print(f"R2: {r2:.4f}")
print("-" * 30)

# 5. 可視化

# (1) 実測値 vs 予測値 (【変更点】日付文字列で色分け)
# 凡例を表示するため、サイズを少し横長にします
plt.figure(figsize=(10, 8))

# テストデータのdate_idを実際の日付オブジェクトに戻す
dates_test_decoded = le_date.inverse_transform(dates_test)
# ユニークな日付を取得（ソート済み）
unique_dates = sorted(list(set(dates_test_decoded)))

# カラーマップを用意（日付の数だけ色を生成）
colors = plt.cm.jet(np.linspace(0, 1, len(unique_dates)))

# 日付ごとにループしてプロット
for i, date_val in enumerate(unique_dates):
    # その日付に該当するデータのインデックス（マスク）を作成
    mask = (dates_test_decoded == date_val)

    # y_test は Series なので .values で numpy 配列としてアクセス
    plt.scatter(y_test.values[mask], y_pred[mask],
                alpha=0.6,
                label=str(date_val), # 凡例用に文字列化
                color=colors[i])

# 理想線 (y=x)
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', label='Perfect Fit')

plt.xlabel('Actual Temperature')
plt.ylabel('Predicted Temperature')
plt.title(f'(R2={r2:.2f})')

# 凡例をグラフの外側に配置
plt.legend(title='Date', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.grid(True)
plt.tight_layout()
plt.savefig('model1_combined_1_actual_vs_pred_colored_date.png')
plt.show()

# (2) 変数重要度 (表表示とグラフ化)
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]

# 変数重要度を表(DataFrame)で作成して表示
df_importances = pd.DataFrame({
    "Feature": [model_cols[i] for i in indices],
    "Importance": importances[indices]
})

print("\n--- Feature Importances (Table) ---")
print(df_importances)
# CSV保存が必要な場合は以下を有効化してください
# df_importances.to_csv('feature_importances.csv', index=False)
print("-" * 30)

top_n = 15
if len(model_cols) < top_n:
    top_n = len(model_cols)

plt.figure(figsize=(12, 6))
plt.bar(range(top_n), importances[indices][:top_n], align="center")
plt.xticks(range(top_n), [model_cols[i] for i in indices][:top_n], rotation=45)
plt.tight_layout()
plt.savefig('model1_combined_2_feature_importance_no_sunshade.png')
plt.show()

# (3) PDP (主要変数)
top_features_indices = indices[:6]
top_features_names = [model_cols[i] for i in top_features_indices]

n_cols_pdp = 3
n_rows_pdp = math.ceil(len(top_features_names) / n_cols_pdp)
fig, ax = plt.subplots(figsize=(12, 4 * n_rows_pdp))
PartialDependenceDisplay.from_estimator(
    rf,
    X_train,
    top_features_indices,
    feature_names=model_cols,
    ax=ax,
    n_cols=n_cols_pdp
)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.savefig('model1_combined_3_pdp_no_sunshade.png')
plt.show()

# (4) 散布図
# sun_shadeをプロット対象から除外
plot_target_cols = ['LandsatLST', 'SVF']
s2_top = [col for col in [model_cols[i] for i in indices] if col in s2_bands][:3]
plot_target_cols.extend(s2_top)

# カラムが存在するものだけプロット
plot_target_cols = [c for c in plot_target_cols if c in df_clean.columns]

num_plots = len(plot_target_cols)
cols = 3
rows = math.ceil(num_plots / cols)
fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
if num_plots > 1:
    axes = axes.flatten()
else:
    axes = [axes]

for i, col in enumerate(plot_target_cols):
    axes[i].scatter(df_clean[col], df_clean[target_col], alpha=0.4, s=20, c='green')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel(target_col)
    axes[i].set_title(f'{col} vs Temp')

    if len(df_clean) > 1:
        try:
            z = np.polyfit(df_clean[col], df_clean[target_col], 1)
            p = np.poly1d(z)
            axes[i].plot(df_clean[col], p(df_clean[col]), "r--")
        except:
            pass

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.savefig('model1_combined_4_scatter_no_sunshade.png')
plt.show()

##変数重要度用の処理

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.inspection import PartialDependenceDisplay
from sklearn.preprocessing import LabelEncoder
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

# 1. データの読み込み
df = pd.read_csv("new_combined_result_with_SVF_and_Bands.csv")

# 2. 前処理

# LandsatLSTの0を欠損値(NaN)に置換
if 'LandsatLST' in df.columns:
    df['LandsatLST'] = df['LandsatLST'].replace(0, np.nan)

# 日付の抽出とカテゴリID化
if '撮影日時' in df.columns:
    df['date_obj'] = pd.to_datetime(df['撮影日時']).dt.date
    le_date = LabelEncoder()
    df['date_id'] = le_date.fit_transform(df['date_obj'])

# sun_shade の数値化 (今回はモデルに使用しませんが、データセットには残しておきます)
if 'Shade' in df.columns:
    if df['Shade'].dtype == 'object':
        le_sun = LabelEncoder()
        df['Shade'] = le_sun.fit_transform(df['Shade'].astype(str))
    else:
        df['Shade'] = pd.to_numeric(df['Shade'], errors='coerce')

# Sentinel-2 バンドのスケール変換
s2_bands = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']
for col in s2_bands:
    if col in df.columns:
        if df[col].max() > 1:
            df[col] = df[col] / 10000.0

# 3. 特徴量エンジニアリング

# ★【変更点】South_NDVI_Shadow_Integrated は作成しない
# ★【変更点】sun_shade も説明変数に含めない

# --- 変数の設定 (初期候補) ---
base_cols = ['LandsatLST', 'NDVI', 'SVF'] # sun_shadeを削除
initial_model_cols = base_cols + s2_bands

target_col = '平均温度__c'

# 欠損値を含む行を削除
available_cols = [c for c in initial_model_cols + [target_col, 'date_id'] if c in df.columns]
df_clean = df.dropna(subset=available_cols)

print(f"分析対象データ数: {len(df_clean)}")

# --- 【変更】VIFによる変数の厳選 (Stepwise VIF Selection) ---
print("\n--- VIFによる変数の厳選を開始 ---")

# 現在のデータフレームにあるカラムのみを対象にする
X_vif_input = df_clean[[c for c in initial_model_cols if c in df_clean.columns]]

# VIFの閾値
vif_threshold = 10.0

# 変数選択ループ
while True:
    # 定数項を加える（VIF計算に必須）
    X_const = add_constant(X_vif_input)

    # VIFを計算
    vif_data = pd.DataFrame()
    vif_data["feature"] = X_const.columns
    vif_data["VIF"] = [variance_inflation_factor(X_const.values, i) for i in range(X_const.shape[1])]

    # 定数項(const)は除外して判定
    vif_data = vif_data[vif_data["feature"] != "const"]

    # 最大VIFを持つ変数を確認
    max_vif = vif_data["VIF"].max()
    max_vif_feature = vif_data.sort_values("VIF", ascending=False).iloc[0]["feature"]

    if max_vif > vif_threshold:
        print(f"削除: {max_vif_feature} (VIF: {max_vif:.2f})")
        # 最もVIFが高い変数を候補から削除
        X_vif_input = X_vif_input.drop(columns=[max_vif_feature])
    else:
        # すべての変数が閾値以下になれば終了
        break

# 厳選された変数のリスト
selected_model_cols = X_vif_input.columns.tolist()
model_cols = selected_model_cols # 以降の処理のために変数名を更新

print("-" * 30)
print("最終的に選択された変数:")
print(model_cols)
print("\n最終的なVIF:")
print(vif_data.sort_values("VIF", ascending=False))
print("-" * 30)

# --- 相関行列の再確認（厳選後） ---
plt.figure(figsize=(10, 8))
correlation_matrix = df_clean[model_cols].corr()
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap='coolwarm', square=True)
plt.title('Correlation Matrix (Selected Features)')
plt.tight_layout()
plt.savefig('model1_correlation_matrix_selected.png')
plt.show()


# 3. モデルの構築 (ランダムフォレスト)

X = df_clean[model_cols]
y = df_clean[target_col]
dates = df_clean['date_id']

# データを8:2に分割
X_train, X_test, y_train, y_test, dates_train, dates_test = train_test_split(
    X, y, dates, test_size=0.2, random_state=42
)

# 学習
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# 4. 検証と評価
y_pred = rf.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("-" * 30)
print(f"モデル評価結果 (厳選変数 - Shade除外):")
print(f"RMSE: {rmse:.4f}")
print(f"R2: {r2:.4f}")
print("-" * 30)

# 5. 可視化

# (1) 実測値 vs 予測値 (日付文字列で色分け)
plt.figure(figsize=(10, 8))

# テストデータのdate_idを実際の日付オブジェクトに戻す
dates_test_decoded = le_date.inverse_transform(dates_test)
unique_dates = sorted(list(set(dates_test_decoded)))
colors = plt.cm.jet(np.linspace(0, 1, len(unique_dates)))

for i, date_val in enumerate(unique_dates):
    mask = (dates_test_decoded == date_val)
    plt.scatter(y_test.values[mask], y_pred[mask],
                alpha=0.6,
                label=str(date_val),
                color=colors[i])

plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', label='Perfect Fit')
plt.xlabel('Actual Temperature')
plt.ylabel('Predicted Temperature')
plt.title(f'Actual vs Predicted (R2={r2:.2f})')
plt.legend(title='Date', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.savefig('model1_combined_1_actual_vs_pred_colored_date.png')
plt.show()

# (2) 変数重要度 (表表示とグラフ化)
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]

# 変数重要度を表(DataFrame)で作成して表示
df_importances = pd.DataFrame({
    "Feature": [model_cols[i] for i in indices],
    "Importance": importances[indices]
})

print("\n--- Feature Importances (Table) ---")
print(df_importances)
# CSV保存が必要な場合は以下を有効化してください
# df_importances.to_csv('feature_importances.csv', index=False)
print("-" * 30)

plt.figure(figsize=(12, 6))
plt.bar(range(len(model_cols)), importances[indices], align="center")
plt.xticks(range(len(model_cols)), [model_cols[i] for i in indices], rotation=45)
plt.tight_layout()
plt.savefig('model1_combined_2_feature_importance_no_sunshade.png')
plt.show()

# (3) PDP (主要変数)
# エラー回避のため、変数が少ない場合の処理を追加
top_n_pdp = min(6, len(model_cols))
top_features_indices = indices[:top_n_pdp]
top_features_names = [model_cols[i] for i in top_features_indices]

if top_n_pdp > 0:
    n_cols_pdp = 3
    n_rows_pdp = math.ceil(len(top_features_names) / n_cols_pdp)
    fig, ax = plt.subplots(figsize=(12, 4 * n_rows_pdp))
    PartialDependenceDisplay.from_estimator(
        rf,
        X_train,
        top_features_indices,
        feature_names=model_cols,
        ax=ax,
        n_cols=n_cols_pdp
    )
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig('model1_combined_3_pdp_no_sunshade.png')
    plt.show()

# (4) 散布図 (重要度上位のみ)
# 重要度上位の変数のみをプロット対象にする
plot_target_cols = [model_cols[i] for i in indices[:min(6, len(model_cols))]]

num_plots = len(plot_target_cols)
if num_plots > 0:
    cols = 3
    rows = math.ceil(num_plots / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))

    if num_plots > 1:
        axes = axes.flatten()
    else:
        axes = [axes]

    for i, col in enumerate(plot_target_cols):
        axes[i].scatter(df_clean[col], df_clean[target_col], alpha=0.4, s=20, c='green')
        axes[i].set_xlabel(col)
        axes[i].set_ylabel(target_col)
        axes[i].set_title(f'{col} vs Temp')

        if len(df_clean) > 1:
            try:
                z = np.polyfit(df_clean[col], df_clean[target_col], 1)
                p = np.poly1d(z)
                axes[i].plot(df_clean[col], p(df_clean[col]), "r--")
            except:
                pass

    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.savefig('model1_combined_4_scatter_no_sunshade.png')
    plt.show()

#モデル２

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.inspection import PartialDependenceDisplay
from sklearn.preprocessing import LabelEncoder
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

# 1. データの読み込み
df = pd.read_csv("new_combined_result_with_SVF_and_Bands.csv")

# 2. 前処理

# LandsatLSTの0を欠損値(NaN)に置換
if 'LandsatLST' in df.columns:
    df['LandsatLST'] = df['LandsatLST'].replace(0, np.nan)

# 日付の抽出とカテゴリID化
if '撮影日時' in df.columns:
    df['date_obj'] = pd.to_datetime(df['撮影日時']).dt.date
    le_date = LabelEncoder()
    df['date_id'] = le_date.fit_transform(df['date_obj'])

# sun_shade の数値化
if 'Shade' in df.columns:
    if df['Shade'].dtype == 'object':
        le_sun = LabelEncoder()
        df['Shade'] = le_sun.fit_transform(df['Shade'].astype(str))
    else:
        df['Shade'] = pd.to_numeric(df['Shade'], errors='coerce')

# Sentinel-2 バンドのスケール変換
s2_bands = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']
for col in s2_bands:
    if col in df.columns:
        if df[col].max() > 1:
            df[col] = df[col] / 10000.0

# 3. 特徴量エンジニアリング

# --- 変数の設定 ---
# ★【変更点】South_NDVI_Shadow_Integrated を削除し、sun_shade を追加
base_cols = ['LandsatLST', 'NDVI', 'Shade', 'SVF']
model_cols = base_cols + s2_bands

target_col = '平均温度__c'

# 欠損値を含む行を削除
available_cols = [c for c in model_cols + [target_col, 'date_id'] if c in df.columns]
df_clean = df.dropna(subset=available_cols)

print(f"分析対象データ数: {len(df_clean)}")

# --- 多重共線性(Multicollinearity)の確認 ---
print("\n--- 多重共線性の確認 ---")

# 相関行列の計算と可視化
plt.figure(figsize=(12, 10))
correlation_matrix = df_clean[model_cols].corr()
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap='coolwarm', square=True)
plt.tight_layout()
plt.savefig('model2_correlation_matrix.png')
plt.show()

# VIF (Variance Inflation Factor) の計算
X_vif = add_constant(df_clean[model_cols])
vif_data = pd.DataFrame()
vif_data["feature"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]

print("\nVariance Inflation Factor (VIF):")
print(vif_data)
print("-" * 30)

# 3. モデルの構築 (ランダムフォレスト)

X = df_clean[model_cols]
y = df_clean[target_col]
dates = df_clean['date_id']

# データを8:2に分割
X_train, X_test, y_train, y_test, dates_train, dates_test = train_test_split(
    X, y, dates, test_size=0.2, random_state=42
)

# 学習
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# 4. 検証と評価
y_pred = rf.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("-" * 30)
print(f"モデル評価結果 (sun_shade使用):")
print(f"RMSE: {rmse:.4f}")
print(f"R2: {r2:.4f}")
print("-" * 30)

# 5. 可視化

# (1) 実測値 vs 予測値 (【変更点】日付文字列で色分け)
# 凡例を表示するため、サイズを少し横長にします
plt.figure(figsize=(10, 8))

# テストデータのdate_idを実際の日付オブジェクトに戻す
dates_test_decoded = le_date.inverse_transform(dates_test)
unique_dates = sorted(list(set(dates_test_decoded)))
colors = plt.cm.jet(np.linspace(0, 1, len(unique_dates)))

for i, date_val in enumerate(unique_dates):
    mask = (dates_test_decoded == date_val)
    plt.scatter(y_test.values[mask], y_pred[mask],
                alpha=0.6,
                label=str(date_val),
                color=colors[i])

plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', label='Perfect Fit')
plt.xlabel('Actual Temperature')
plt.ylabel('Predicted Temperature')
plt.title(f'(R2={r2:.2f})')
plt.legend(title='Date', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.savefig('model2_combined_1_actual_vs_pred_colored_date.png')
plt.show()

# (2) 変数重要度 (表表示とグラフ化)
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]

# ★★★ 追加部分: 変数重要度を表(DataFrame)で作成して表示 ★★★
df_importances = pd.DataFrame({
    "Feature": [model_cols[i] for i in indices],
    "Importance": importances[indices]
})

print("\n--- Feature Importances (Table) ---")
print(df_importances)
# CSV保存が必要な場合は以下を有効化してください
# df_importances.to_csv('feature_importances_model2.csv', index=False)
print("-" * 30)
# ★★★ 追加部分終了 ★★★

top_n = 15
if len(model_cols) < top_n:
    top_n = len(model_cols)

plt.figure(figsize=(12, 6))
plt.bar(range(top_n), importances[indices][:top_n], align="center")
plt.xticks(range(top_n), [model_cols[i] for i in indices][:top_n], rotation=45)
plt.tight_layout()
plt.savefig('model2_combined_2_feature_importance.png')
plt.show()

# (3) PDP (主要変数)
top_features_indices = indices[:6]
top_features_names = [model_cols[i] for i in top_features_indices]

n_cols_pdp = 3
n_rows_pdp = math.ceil(len(top_features_names) / n_cols_pdp)
fig, ax = plt.subplots(figsize=(12, 4 * n_rows_pdp))
PartialDependenceDisplay.from_estimator(
    rf,
    X_train,
    top_features_indices,
    feature_names=model_cols,
    ax=ax,
    n_cols=n_cols_pdp
)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.savefig('model2_combined_3_pdp.png')
plt.show()

# (4) 散布図
plot_target_cols = ['LandsatLST', 'Shade', 'SVF']
s2_top = [col for col in [model_cols[i] for i in indices] if col in s2_bands][:3]
plot_target_cols.extend(s2_top)

plot_target_cols = [c for c in plot_target_cols if c in df_clean.columns]

num_plots = len(plot_target_cols)
cols = 3
rows = math.ceil(num_plots / cols)
fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
if num_plots > 1:
    axes = axes.flatten()
else:
    axes = [axes]

for i, col in enumerate(plot_target_cols):
    axes[i].scatter(df_clean[col], df_clean[target_col], alpha=0.4, s=20, c='green')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel(target_col)
    axes[i].set_title(f'{col} vs Temp')

    if len(df_clean) > 1:
        try:
            z = np.polyfit(df_clean[col], df_clean[target_col], 1)
            p = np.poly1d(z)
            axes[i].plot(df_clean[col], p(df_clean[col]), "r--")
        except:
            pass

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.savefig('model2_combined_4_scatter.png')
plt.show()

##変数重要度用の処理

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.inspection import PartialDependenceDisplay
from sklearn.preprocessing import LabelEncoder
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

# 1. データの読み込み
df = pd.read_csv("new_combined_result_with_SVF_and_Bands.csv")

# 2. 前処理

# LandsatLSTの0を欠損値(NaN)に置換
if 'LandsatLST' in df.columns:
    df['LandsatLST'] = df['LandsatLST'].replace(0, np.nan)

# 日付の抽出とカテゴリID化
if '撮影日時' in df.columns:
    df['date_obj'] = pd.to_datetime(df['撮影日時']).dt.date
    le_date = LabelEncoder()
    df['date_id'] = le_date.fit_transform(df['date_obj'])

# sun_shade の数値化
if 'Shade' in df.columns:
    if df['Shade'].dtype == 'object':
        le_sun = LabelEncoder()
        df['Shade'] = le_sun.fit_transform(df['Shade'].astype(str))
    else:
        df['Shade'] = pd.to_numeric(df['Shade'], errors='coerce')

# Sentinel-2 バンドのスケール変換
s2_bands = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']
for col in s2_bands:
    if col in df.columns:
        if df[col].max() > 1:
            df[col] = df[col] / 10000.0

# 3. 特徴量エンジニアリング

# --- 変数の設定 (初期候補) ---
# ★【変更点】South_NDVI_Shadow_Integrated を削除し、sun_shade を追加
base_cols = ['LandsatLST', 'NDVI', 'Shade', 'SVF']
initial_model_cols = base_cols + s2_bands

target_col = '平均温度__c'

# 欠損値を含む行を削除
available_cols = [c for c in initial_model_cols + [target_col, 'date_id'] if c in df.columns]
df_clean = df.dropna(subset=available_cols)

print(f"分析対象データ数: {len(df_clean)}")

# --- 【変更】VIFによる変数の厳選 (Stepwise VIF Selection) ---
print("\n--- VIFによる変数の厳選を開始 ---")

# 現在のデータフレームにあるカラムのみを対象にする
X_vif_input = df_clean[[c for c in initial_model_cols if c in df_clean.columns]]

# VIFの閾値
vif_threshold = 10.0

# 変数選択ループ
while True:
    # 定数項を加える（VIF計算に必須）
    X_const = add_constant(X_vif_input)

    # VIFを計算
    vif_data = pd.DataFrame()
    vif_data["feature"] = X_const.columns
    vif_data["VIF"] = [variance_inflation_factor(X_const.values, i) for i in range(X_const.shape[1])]

    # 定数項(const)は除外して判定
    vif_data = vif_data[vif_data["feature"] != "const"]

    # 最大VIFを持つ変数を確認
    max_vif = vif_data["VIF"].max()
    max_vif_feature = vif_data.sort_values("VIF", ascending=False).iloc[0]["feature"]

    if max_vif > vif_threshold:
        print(f"削除: {max_vif_feature} (VIF: {max_vif:.2f})")
        # 最もVIFが高い変数を候補から削除
        X_vif_input = X_vif_input.drop(columns=[max_vif_feature])
    else:
        # すべての変数が閾値以下になれば終了
        break

# 厳選された変数のリスト
selected_model_cols = X_vif_input.columns.tolist()
model_cols = selected_model_cols # 以降の処理のために変数名を更新

print("-" * 30)
print("最終的に選択された変数:")
print(model_cols)
print("\n最終的なVIF:")
print(vif_data.sort_values("VIF", ascending=False))
print("-" * 30)

# --- 相関行列の再確認（厳選後） ---
plt.figure(figsize=(10, 8))
correlation_matrix = df_clean[model_cols].corr()
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap='coolwarm', square=True)
plt.title('Correlation Matrix (Selected Features)')
plt.tight_layout()
plt.savefig('model2_correlation_matrix_selected.png')
plt.show()


# 3. モデルの構築 (ランダムフォレスト)

X = df_clean[model_cols]
y = df_clean[target_col]
dates = df_clean['date_id']

# データを8:2に分割
X_train, X_test, y_train, y_test, dates_train, dates_test = train_test_split(
    X, y, dates, test_size=0.2, random_state=42
)

# 学習
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# 4. 検証と評価
y_pred = rf.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("-" * 30)
print(f"モデル評価結果 (厳選変数 - sun_shade使用):")
print(f"RMSE: {rmse:.4f}")
print(f"R2: {r2:.4f}")
print("-" * 30)

# 5. 可視化

# (1) 実測値 vs 予測値 (日付文字列で色分け)
plt.figure(figsize=(10, 8))

# テストデータのdate_idを実際の日付オブジェクトに戻す
dates_test_decoded = le_date.inverse_transform(dates_test)
unique_dates = sorted(list(set(dates_test_decoded)))
colors = plt.cm.jet(np.linspace(0, 1, len(unique_dates)))

for i, date_val in enumerate(unique_dates):
    mask = (dates_test_decoded == date_val)
    plt.scatter(y_test.values[mask], y_pred[mask],
                alpha=0.6,
                label=str(date_val),
                color=colors[i])

plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', label='Perfect Fit')
plt.xlabel('Actual Temperature')
plt.ylabel('Predicted Temperature')
plt.title(f'Actual vs Predicted (R2={r2:.2f})')
plt.legend(title='Date', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.savefig('model2_combined_1_actual_vs_pred_colored_date.png')
plt.show()

# (2) 変数重要度 (表表示とグラフ化)
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]

# ★★★ 追加部分: 変数重要度を表(DataFrame)で作成して表示 ★★★
df_importances = pd.DataFrame({
    "Feature": [model_cols[i] for i in indices],
    "Importance": importances[indices]
})

print("\n--- Feature Importances (Table) ---")
print(df_importances)
# CSV保存が必要な場合は以下を有効化してください
# df_importances.to_csv('feature_importances_model2.csv', index=False)
print("-" * 30)
# ★★★ 追加部分終了 ★★★

plt.figure(figsize=(12, 6))
plt.bar(range(len(model_cols)), importances[indices], align="center")
plt.xticks(range(len(model_cols)), [model_cols[i] for i in indices], rotation=45)
plt.tight_layout()
plt.savefig('model2_combined_2_feature_importance.png')
plt.show()

# (3) PDP (主要変数)
# エラー回避のため、変数が少ない場合の処理を追加
top_n_pdp = min(6, len(model_cols))
top_features_indices = indices[:top_n_pdp]
top_features_names = [model_cols[i] for i in top_features_indices]

if top_n_pdp > 0:
    n_cols_pdp = 3
    n_rows_pdp = math.ceil(len(top_features_names) / n_cols_pdp)
    fig, ax = plt.subplots(figsize=(12, 4 * n_rows_pdp))
    PartialDependenceDisplay.from_estimator(
        rf,
        X_train,
        top_features_indices,
        feature_names=model_cols,
        ax=ax,
        n_cols=n_cols_pdp
    )
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig('model2_combined_3_pdp.png')
    plt.show()

# (4) 散布図 (重要度上位のみ)
# 重要度上位の変数のみをプロット対象にする
plot_target_cols = [model_cols[i] for i in indices[:min(6, len(model_cols))]]

num_plots = len(plot_target_cols)
if num_plots > 0:
    cols = 3
    rows = math.ceil(num_plots / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))

    if num_plots > 1:
        axes = axes.flatten()
    else:
        axes = [axes]

    for i, col in enumerate(plot_target_cols):
        axes[i].scatter(df_clean[col], df_clean[target_col], alpha=0.4, s=20, c='green')
        axes[i].set_xlabel(col)
        axes[i].set_ylabel(target_col)
        axes[i].set_title(f'{col} vs Temp')

        if len(df_clean) > 1:
            try:
                z = np.polyfit(df_clean[col], df_clean[target_col], 1)
                p = np.poly1d(z)
                axes[i].plot(df_clean[col], p(df_clean[col]), "r--")
            except:
                pass

    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.savefig('model2_combined_4_scatter.png')
    plt.show()

#モデル３

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.inspection import PartialDependenceDisplay
from sklearn.preprocessing import LabelEncoder
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

# 1. データの読み込み
df = pd.read_csv("new_combined_result_with_SVF_and_Bands.csv")

# 2. 前処理

# LandsatLSTの0を欠損値(NaN)に置換
if 'LandsatLST' in df.columns:
    df['LandsatLST'] = df['LandsatLST'].replace(0, np.nan)

# 日付の抽出とカテゴリID化
if '撮影日時' in df.columns:
    df['date_obj'] = pd.to_datetime(df['撮影日時']).dt.date
    le_date = LabelEncoder()
    df['date_id'] = le_date.fit_transform(df['date_obj'])

# sun_shade の数値化
if 'Shade' in df.columns:
    if df['Shade'].dtype == 'object':
        le_sun = LabelEncoder()
        df['Shade'] = le_sun.fit_transform(df['Shade'].astype(str))
    else:
        df['Shade'] = pd.to_numeric(df['Shade'], errors='coerce')

# Sentinel-2 バンドのスケール変換
s2_bands = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']
for col in s2_bands:
    if col in df.columns:
        if df[col].max() > 1:
            df[col] = df[col] / 10000.0

# 3. 特徴量エンジニアリング

# 南側植生と日陰の統合変数を作成
if 'Shade_NDVI' in df.columns and 'Shade' in df.columns:
    df['Shade NDVI'] = np.maximum(df['Shade_NDVI'], 1 - df['Shade'])

# --- 変数の設定 ---
# ★【変更点】'ShikisaiLST' を削除しました
base_cols = ['LandsatLST', 'NDVI', 'Shade NDVI', 'SVF']
model_cols = base_cols + s2_bands

target_col = '平均温度__c'

# 欠損値を含む行を削除
available_cols = [c for c in model_cols + [target_col, 'date_id'] if c in df.columns]
df_clean = df.dropna(subset=available_cols)

print(f"分析対象データ数: {len(df_clean)}")

# --- 多重共線性(Multicollinearity)の確認 ---
print("\n--- 多重共線性の確認 ---")

# 相関行列の計算と可視化
plt.figure(figsize=(12, 10))
correlation_matrix = df_clean[model_cols].corr()
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap='coolwarm', square=True)
plt.tight_layout()
plt.savefig('model3_correlation_matrix.png')
plt.show()

# VIF (Variance Inflation Factor) の計算
X_vif = add_constant(df_clean[model_cols])
vif_data = pd.DataFrame()
vif_data["feature"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]

print("\nVariance Inflation Factor (VIF):")
print(vif_data)
print("-" * 30)


# 3. モデルの構築 (ランダムフォレスト)

X = df_clean[model_cols]
y = df_clean[target_col]
dates = df_clean['date_id']

# データを8:2に分割
X_train, X_test, y_train, y_test, dates_train, dates_test = train_test_split(
    X, y, dates, test_size=0.2, random_state=42
)

# 学習
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# 4. 検証と評価
y_pred = rf.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("-" * 30)
print(f"モデル評価結果 (ShikisaiLST除外):")
print(f"RMSE: {rmse:.4f}")
print(f"R2: {r2:.4f}")
print("-" * 30)

# 5. 可視化

# (1) 実測値 vs 予測値 (【変更点】日付文字列で色分け)
plt.figure(figsize=(10, 8))

# テストデータのdate_idを実際の日付オブジェクトに戻す
dates_test_decoded = le_date.inverse_transform(dates_test)
unique_dates = sorted(list(set(dates_test_decoded)))
colors = plt.cm.jet(np.linspace(0, 1, len(unique_dates)))

for i, date_val in enumerate(unique_dates):
    mask = (dates_test_decoded == date_val)
    plt.scatter(y_test.values[mask], y_pred[mask],
                alpha=0.6,
                label=str(date_val),
                color=colors[i])

plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', label='Perfect Fit')
plt.xlabel('Actual Temperature')
plt.ylabel('Predicted Temperature')
plt.title(f'(R2={r2:.2f})')
plt.legend(title='Date', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.savefig('model3_combined_1_actual_vs_pred_colored_date.png')
plt.show()

# (2) 変数重要度 (表表示とグラフ化)
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]

# ★★★ 追加部分: 変数重要度を表(DataFrame)で作成して表示 ★★★
df_importances = pd.DataFrame({
    "Feature": [model_cols[i] for i in indices],
    "Importance": importances[indices]
})

print("\n--- Feature Importances (Table) ---")
print(df_importances)
# CSV保存が必要な場合は以下を有効化してください
# df_importances.to_csv('feature_importances_model3.csv', index=False)
print("-" * 30)
# ★★★ 追加部分終了 ★★★

top_n = 15
if len(model_cols) < top_n:
    top_n = len(model_cols)

plt.figure(figsize=(12, 6))
plt.bar(range(top_n), importances[indices][:top_n], align="center")
plt.xticks(range(top_n), [model_cols[i] for i in indices][:top_n], rotation=45)
plt.tight_layout()
plt.savefig('model3_combined_2_feature_importance.png')
plt.show()

# (3) PDP (主要変数)
top_features_indices = indices[:6]
top_features_names = [model_cols[i] for i in top_features_indices]

n_cols_pdp = 3
n_rows_pdp = math.ceil(len(top_features_names) / n_cols_pdp)
fig, ax = plt.subplots(figsize=(12, 4 * n_rows_pdp))
PartialDependenceDisplay.from_estimator(
    rf,
    X_train,
    top_features_indices,
    feature_names=model_cols,
    ax=ax,
    n_cols=n_cols_pdp
)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.savefig('model3_combined_3_pdp.png')
plt.show()

# (4) 散布図
plot_target_cols = ['LandsatLST', 'Shade NDVI', 'SVF']
s2_top = [col for col in [model_cols[i] for i in indices] if col in s2_bands][:3]
plot_target_cols.extend(s2_top)

plot_target_cols = [c for c in plot_target_cols if c in df_clean.columns]

num_plots = len(plot_target_cols)
cols = 3
rows = math.ceil(num_plots / cols)
fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
if num_plots > 1:
    axes = axes.flatten()
else:
    axes = [axes]

for i, col in enumerate(plot_target_cols):
    axes[i].scatter(df_clean[col], df_clean[target_col], alpha=0.4, s=20, c='green')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel(target_col)
    axes[i].set_title(f'{col} vs Temp')

    if len(df_clean) > 1:
        try:
            z = np.polyfit(df_clean[col], df_clean[target_col], 1)
            p = np.poly1d(z)
            axes[i].plot(df_clean[col], p(df_clean[col]), "r--")
        except:
            pass

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.savefig('model3_combined_4_scatter.png')
plt.show()

##変数重要度用の処理

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.inspection import PartialDependenceDisplay
from sklearn.preprocessing import LabelEncoder
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

# 1. データの読み込み
# ※ファイルパスは環境に合わせて変更してください
df = pd.read_csv("new_combined_result_with_SVF_and_Bands.csv")

# 2. 前処理

# LandsatLSTの0を欠損値(NaN)に置換
if 'LandsatLST' in df.columns:
    df['LandsatLST'] = df['LandsatLST'].replace(0, np.nan)

# 日付の抽出とカテゴリID化
if '撮影日時' in df.columns:
    df['date_obj'] = pd.to_datetime(df['撮影日時']).dt.date
    le_date = LabelEncoder()
    df['date_id'] = le_date.fit_transform(df['date_obj'])

# sun_shade の数値化
if 'Shade' in df.columns:
    if df['Shade'].dtype == 'object':
        le_sun = LabelEncoder()
        df['Shade'] = le_sun.fit_transform(df['Shade'].astype(str))
    else:
        df['Shade'] = pd.to_numeric(df['Shade'], errors='coerce')

# Sentinel-2 バンドのスケール変換
s2_bands = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']
for col in s2_bands:
    if col in df.columns:
        if df[col].max() > 1:
            df[col] = df[col] / 10000.0

# 3. 特徴量エンジニアリング

# 南側植生と日陰の統合変数を作成
if 'Shade_NDVI' in df.columns and 'Shade' in df.columns:
    df['Shade NDVI'] = np.maximum(df['Shade_NDVI'], 1 - df['Shade'])

# --- 変数の設定 (初期候補) ---
base_cols = ['LandsatLST', 'NDVI', 'Shade NDVI', 'SVF']
initial_model_cols = base_cols + s2_bands
target_col = '平均温度__c'

# 欠損値を含む行を削除
# 候補変数がデータに含まれているか確認
available_cols = [c for c in initial_model_cols + [target_col, 'date_id'] if c in df.columns]
df_clean = df.dropna(subset=available_cols)

print(f"分析対象データ数: {len(df_clean)}")

# --- 【追加】VIFによる変数の厳選 (Stepwise VIF Selection) ---
print("\n--- VIFによる変数の厳選を開始 ---")

# 現在のデータフレームにあるカラムのみを対象にする
X_vif_input = df_clean[[c for c in initial_model_cols if c in df_clean.columns]]

# VIFの閾値 (一般的に10以上は多重共線性の疑いあり)
vif_threshold = 10.0

# 変数選択ループ
while True:
    # 定数項を加える（VIF計算に必須）
    X_const = add_constant(X_vif_input)

    # VIFを計算
    vif_data = pd.DataFrame()
    vif_data["feature"] = X_const.columns
    vif_data["VIF"] = [variance_inflation_factor(X_const.values, i) for i in range(X_const.shape[1])]

    # 定数項(const)は除外して判定
    vif_data = vif_data[vif_data["feature"] != "const"]

    # 最大VIFを持つ変数を確認
    max_vif = vif_data["VIF"].max()
    max_vif_feature = vif_data.sort_values("VIF", ascending=False).iloc[0]["feature"]

    if max_vif > vif_threshold:
        print(f"削除: {max_vif_feature} (VIF: {max_vif:.2f})")
        # 最もVIFが高い変数を候補から削除
        X_vif_input = X_vif_input.drop(columns=[max_vif_feature])
    else:
        # すべての変数が閾値以下になれば終了
        break

# 厳選された変数のリスト
selected_model_cols = X_vif_input.columns.tolist()

print("-" * 30)
print("最終的に選択された変数:")
print(selected_model_cols)
print("\n最終的なVIF:")
print(vif_data.sort_values("VIF", ascending=False))
print("-" * 30)

# モデル構築用に変数を更新
model_cols = selected_model_cols


# --- 相関行列の再確認（厳選後） ---
plt.figure(figsize=(10, 8))
correlation_matrix = df_clean[model_cols].corr()
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap='coolwarm', square=True)
plt.title("Correlation Matrix (Selected Features)")
plt.tight_layout()
plt.savefig('model3_correlation_matrix_selected.png')
plt.show()


# 4. モデルの構築 (ランダムフォレスト)

X = df_clean[model_cols]
y = df_clean[target_col]
dates = df_clean['date_id']

# データを8:2に分割
X_train, X_test, y_train, y_test, dates_train, dates_test = train_test_split(
    X, y, dates, test_size=0.2, random_state=42
)

# 学習
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# 5. 検証と評価
y_pred = rf.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("-" * 30)
print(f"モデル評価結果 (厳選変数):")
print(f"RMSE: {rmse:.4f}")
print(f"R2: {r2:.4f}")
print("-" * 30)

# 6. 可視化

# (1) 実測値 vs 予測値 (日付文字列で色分け)
plt.figure(figsize=(10, 8))

# テストデータのdate_idを実際の日付オブジェクトに戻す
dates_test_decoded = le_date.inverse_transform(dates_test)
unique_dates = sorted(list(set(dates_test_decoded)))
colors = plt.cm.jet(np.linspace(0, 1, len(unique_dates)))

for i, date_val in enumerate(unique_dates):
    mask = (dates_test_decoded == date_val)
    plt.scatter(y_test.values[mask], y_pred[mask],
                alpha=0.6,
                label=str(date_val),
                color=colors[i])

plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', label='Perfect Fit')
plt.xlabel('Actual Temperature')
plt.ylabel('Predicted Temperature')
plt.title(f'Actual vs Predicted (R2={r2:.2f})')
plt.legend(title='Date', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.savefig('model3_combined_1_actual_vs_pred_colored_date.png')
plt.show()

# (2) 変数重要度 (表表示とグラフ化)
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]

# 変数重要度を表(DataFrame)で作成して表示
df_importances = pd.DataFrame({
    "Feature": [model_cols[i] for i in indices],
    "Importance": importances[indices]
})

print("\n--- Feature Importances (Table) ---")
print(df_importances)
print("-" * 30)

# 重要度グラフ
plt.figure(figsize=(12, 6))
plt.bar(range(len(model_cols)), importances[indices], align="center")
plt.xticks(range(len(model_cols)), [model_cols[i] for i in indices], rotation=45)
plt.tight_layout()
plt.savefig('model3_combined_2_feature_importance.png')
plt.show()

# (3) PDP (主要変数)
# 変数が減っている可能性があるため、エラー回避のため数を調整
top_n_pdp = min(6, len(model_cols))
top_features_indices = indices[:top_n_pdp]
top_features_names = [model_cols[i] for i in top_features_indices]

if top_n_pdp > 0:
    n_cols_pdp = 3
    n_rows_pdp = math.ceil(len(top_features_names) / n_cols_pdp)
    fig, ax = plt.subplots(figsize=(12, 4 * n_rows_pdp))
    PartialDependenceDisplay.from_estimator(
        rf,
        X_train,
        top_features_indices,
        feature_names=model_cols,
        ax=ax,
        n_cols=n_cols_pdp
    )
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig('model3_combined_3_pdp.png')
    plt.show()

# (4) 散布図 (重要度上位のみ)
# 重要度上位の変数とターゲットの散布図を描画
plot_target_cols = [model_cols[i] for i in indices[:min(6, len(model_cols))]]

num_plots = len(plot_target_cols)
if num_plots > 0:
    cols = 3
    rows = math.ceil(num_plots / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))

    if num_plots > 1:
        axes = axes.flatten()
    else:
        axes = [axes] # 1つの場合リスト化

    for i, col in enumerate(plot_target_cols):
        axes[i].scatter(df_clean[col], df_clean[target_col], alpha=0.4, s=20, c='green')
        axes[i].set_xlabel(col)
        axes[i].set_ylabel(target_col)
        axes[i].set_title(f'{col} vs Temp')

        if len(df_clean) > 1:
            try:
                z = np.polyfit(df_clean[col], df_clean[target_col], 1)
                p = np.poly1d(z)
                axes[i].plot(df_clean[col], p(df_clean[col]), "r--")
            except:
                pass

    # 余った枠を消す
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.savefig('model3_combined_4_scatter.png')
    plt.show()

#LST分布

In [ ]:
!pip install cartopy
!pip install geedim

In [ ]:
import os
import glob
import time
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import pandas as pd
import random
import numpy as np

import ee
import geemap
from geemap import cartoee
import geemap.colormaps as cm
import cartopy.crs as ccrs

import rasterio
from rasterio.plot import show

import ipywidgets as widgets
from IPython.display import display

from sklearn.linear_model import LinearRegression

from datetime import datetime, timedelta


%matplotlib inline

In [ ]:
try:
    YOUR_PROJECT = "dogwood-vision-457006-q8"
    geemap.ee_initialize(project=YOUR_PROJECT)
except Exception as e:
    print(e)
    print("Google Earth Engineの初期化に失敗しました。認証プロセスを確認してください。")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split  # 追加

train_file = "new_combined_result_with_SVF_and_Bands.csv" # 学習用（ラベルあり）
predict_file = "data_with_svf_and_s2_bands_20250908.csv"  # 予測対象（ラベルなし）
output_file = "predicted_result_20250908.csv"

s2_bands = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']
base_cols = ['LandsatLST', 'NDVI', 'Shade NDVI', 'SVF']
model_cols = base_cols + s2_bands
target_col = '平均温度__c'

def preprocess_df(df):
    """ご提示のロジックに基づく前処理関数"""
    df = df.copy()

    # LandsatLSTの0をNaNに置換
    if 'LandsatLST' in df.columns:
        df['LandsatLST'] = df['LandsatLST'].replace(0, np.nan)

    # 日付の数値化 (date_id)
    if '撮影日時' in df.columns:
        df['date_obj'] = pd.to_datetime(df['撮影日時']).dt.date
        le_date = LabelEncoder()
        df['date_id'] = le_date.fit_transform(df['date_obj'])

    # sun_shade の数値化
    if 'Shade' in df.columns:
        if df['Shade'].dtype == 'object':
            le_sun = LabelEncoder()
            df['Shade'] = le_sun.fit_transform(df['Shade'].astype(str))
        else:
            df['Shade'] = pd.to_numeric(df['Shade'], errors='coerce')

    # Sentinel-2 バンドのスケール変換
    for col in s2_bands:
        if col in df.columns:
            if df[col].max() > 1:
                df[col] = df[col] / 10000.0

    # 南側植生と日陰の統合変数を作成
    if 'Shade_NDVI' in df.columns and 'Shade' in df.columns:
        df['Shade NDVI'] = np.maximum(df['Shade_NDVI'], 1 - df['Shade'])

    return df

# 1. モデルの学習 (学習用データを使用)
print(f"Loading training data: {train_file}")
df_train_raw = pd.read_csv(train_file)
df_train = preprocess_df(df_train_raw)

# 欠損値を除去して学習
available_cols_train = [c for c in model_cols + [target_col] if c in df_train.columns]
df_train_clean = df_train.dropna(subset=available_cols_train)

X_all = df_train_clean[model_cols]
y_all = df_train_clean[target_col]

# データを8:2に分割 (random_state=42)
# test_size=0.2 で2割を検証用に、残りの8割を学習用(X_train, y_train)にします
X_train, X_val, y_train, y_val = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42
)

print(f"Training model with {len(X_train)} samples (Training split)...")
print(f"Validation set size: {len(X_val)} samples")

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# 学習に使用しなかった2割のデータで精度を確認（オプション）
val_score = rf.score(X_val, y_val)
print(f"Validation R^2 Score: {val_score:.4f}")

# 2. 予測の実行 (アップロードされたファイルを使用)
print(f"Loading prediction target: {predict_file}")
df_predict_raw = pd.read_csv(predict_file)
df_predict = preprocess_df(df_predict_raw)

# 予測に必要な特徴量が揃っているか確認
X_predict = df_predict[model_cols]

# 予測実行 (特徴量に欠損がある行は NaN を返すように制御)
mask = X_predict.notna().all(axis=1)
predictions = np.full(len(df_predict), np.nan)
if mask.any():
    predictions[mask] = rf.predict(X_predict[mask])

# 結果を結合して保存
df_predict_raw['Predicted_Temp'] = predictions
df_predict_raw.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"Prediction completed. Results saved to: {output_file}")
print(f"Total predicted rows: {mask.sum()} / {len(df_predict)}")

# (オプション) 予測結果の分布を確認
if mask.any():
    plt.figure(figsize=(8, 5))
    plt.hist(predictions[mask], bins=30, color='skyblue', edgecolor='black')
    plt.title('Distribution of Predicted Temperatures')
    plt.xlabel('Predicted Temperature')
    plt.ylabel('Frequency')
    plt.grid(axis='y', alpha=0.3)
    plt.savefig('prediction_distribution.png')
    # plt.show() # 環境によってはコメントアウト推奨

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.io.img_tiles as cimgt
import matplotlib.patches as mpatches
import matplotlib.patheffects as PathEffects
import numpy as np
from matplotlib.colors import Normalize

# ==========================================
# 1. データの読み込み
# ==========================================
file_path = "predicted_result_20250908.csv"
df = pd.read_csv(file_path)
df_plot = df.dropna(subset=['Predicted_Temp', 'latitude', 'longitude'])

# マップ範囲の計算
min_lon, max_lon = df_plot['longitude'].min(), df_plot['longitude'].max()
min_lat, max_lat = df_plot['latitude'].min(), df_plot['latitude'].max()
mid_lon = (min_lon + max_lon) / 2

# ==========================================
# 2. 設定：CartoDB "Voyager" (濃いめ)
# ==========================================
class CartoDBVoyager(cimgt.GoogleTiles):
    def _image_url(self, tile):
        x, y, z = tile
        return f'https://a.basemaps.cartocdn.com/rastertiles/voyager/{z}/{x}/{y}.png'

request = CartoDBVoyager()

norm = Normalize(vmin=35, vmax=50)
cmap = plt.get_cmap('jet')
zoom_level = 17

# ==========================================
# 3. ヘルパー関数（方位記号・スケールバー）
# ==========================================
def add_scalebar_and_north_arrow(ax, map_extent, length_km=1.0):
    min_lon, max_lon, min_lat, max_lat = map_extent
    pe_text = [PathEffects.withStroke(linewidth=3, foreground="white")]
    pe_line = [PathEffects.withStroke(linewidth=4, foreground="white")]

    # 1. 方位記号 (左上)
    arrow_x, arrow_y = 0.08, 0.92
    ax.text(arrow_x, arrow_y, 'N', transform=ax.transAxes,
            ha='center', va='bottom', fontsize=16, fontweight='bold',
            zorder=200, path_effects=pe_text)
    ax.plot([arrow_x, arrow_x], [arrow_y - 0.08, arrow_y - 0.02],
            transform=ax.transAxes, color='black', linewidth=3,
            zorder=200, path_effects=pe_line)
    ax.plot(arrow_x, arrow_y - 0.01, transform=ax.transAxes,
            marker='^', markersize=15, color='black',
            markeredgecolor='white', markeredgewidth=1.5, zorder=201)

    # 2. スケールバー (右下)
    lon_span = max_lon - min_lon
    lat_span = max_lat - min_lat
    sb_right_lon = min_lon + lon_span * 0.95
    sb_base_lat = min_lat + lat_span * 0.05

    center_lat = np.mean([min_lat, max_lat])
    meters_per_degree_lon = 111320 * np.cos(np.radians(center_lat))
    length_deg = (length_km * 1000) / meters_per_degree_lon
    sb_left_lon = sb_right_lon - length_deg
    bar_height = lat_span * 0.015

    ax.plot([sb_left_lon, sb_right_lon], [sb_base_lat, sb_base_lat],
            color='black', linewidth=3, transform=ccrs.PlateCarree(),
            zorder=200, path_effects=pe_line)
    ax.plot([sb_left_lon, sb_left_lon], [sb_base_lat, sb_base_lat + bar_height],
            color='black', linewidth=3, transform=ccrs.PlateCarree(),
            zorder=200, path_effects=pe_line)
    ax.plot([sb_right_lon, sb_right_lon], [sb_base_lat, sb_base_lat + bar_height],
            color='black', linewidth=3, transform=ccrs.PlateCarree(),
            zorder=200, path_effects=pe_line)

    label_text = f'{length_km} km' if length_km >= 1 else f'{int(length_km*1000)} m'
    ax.text((sb_left_lon + sb_right_lon) / 2, sb_base_lat + bar_height * 0.6,
            label_text, ha='center', va='bottom', fontsize=11, fontweight='bold',
            transform=ccrs.PlateCarree(), zorder=200, path_effects=pe_text)

# ==========================================
# 4. 作図関数（タイトル削除版）
# ==========================================
def create_split_map(extent, filename):
    fig = plt.figure(figsize=(12, 10))
    ax = plt.axes(projection=request.crs)

    ax.set_extent(extent, crs=ccrs.PlateCarree())
    ax.add_image(request, zoom_level)

    scatter = ax.scatter(
        df_plot['longitude'],
        df_plot['latitude'],
        c=df_plot['Predicted_Temp'],
        cmap=cmap,
        norm=norm,
        s=50,
        edgecolors='black',
        linewidth=0.5,
        alpha=0.9,
        transform=ccrs.PlateCarree(),
        zorder=20
    )

    add_scalebar_and_north_arrow(ax, extent, length_km=0.5)

    credit_text = 'Map tiles by Carto (Voyager), under CC BY 3.0. Data by OpenStreetMap, under ODbL.'
    ax.text(0.99, 0.005, credit_text,
            transform=ax.transAxes, ha='right', va='bottom', fontsize=7,
            bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'),
            zorder=100)

    cbar = plt.colorbar(scatter, ax=ax, orientation='vertical', shrink=0.7, pad=0.05)
    cbar.set_label('Predicted Temperature (℃)', fontsize=12)

    # --- 変更点: plt.title() を削除しました ---

    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved: {filename}")

# ==========================================
# 5. 実行
# ==========================================
lon_buffer = 0.001
lat_buffer = 0.001

# 左半分
extent_west = [min_lon - lon_buffer, mid_lon, min_lat - lat_buffer, max_lat + lat_buffer]
create_split_map(extent_west, 'map_1_west_voyager_no_title.png')

# 右半分
extent_east = [mid_lon, max_lon + lon_buffer, min_lat - lat_buffer, max_lat + lat_buffer]
create_split_map(extent_east, 'map_2_east_voyager_no_title.png')